# 13 — DuckLake Catalog Backend

`DuckLakeCatalog` (`ailake-catalog`, feature `catalog-ducklake`) is an alternative
`CatalogProvider` that stores AI-Lake table metadata in a real **DuckLake** catalog,
driven entirely through the real DuckDB `ducklake` extension — no hand-rolled catalog
DDL. It sits alongside `HadoopCatalog` (file-based, the default used by every other
notebook in this demo), `JdbcCatalog`, `GlueCatalog`, and `NessieCatalog`.

Full design doc: `docs/guides/DUCKLAKE_CATALOG.md`.

| | |
|---|---|
| **Selectable from** | `ailake-cli` only (`--catalog ducklake`) — **not** exposed in `ailake-py`, so this notebook drives the CLI binary via `subprocess` instead of `import ailake` |
| **Store requirement** | Local filesystem only (no `s3://`, `gs://`, `az://`) |
| **Ownership split** | DuckLake owns schemas/tables/columns/snapshots/row content; a DuckDB sidecar (`main.ailake_vector_index`, plain tables, no DuckLake versioning) owns everything DuckLake's fixed schema has no room for — centroid, radius, HNSW offset/length, index status |

```
DuckDB connection
├── main.*   — AI-Lake sidecar: ailake_tables, ailake_table_props,
│              ailake_vector_index, ailake_equality_deletes
└── ATTACH 'ducklake:<meta.db>' AS lake (DATA_PATH '<warehouse>/data')
    └── lake.<namespace>.<table>   — real DuckLake-managed table
```

This notebook: `create` → seed a Parquet part-file via the Python SDK (against a
throwaway `HadoopCatalog` path, just to get valid AI-Lake bytes) → `insert` it into
a DuckLake-catalog table via the CLI → `search` → `evolve` (add a column) → `insert`
again → `compact` → `info` → then open **both** halves of the catalog directly with
`duckdb` to make the sidecar/DuckLake split concrete.


## 0. Setup — locate the `ailake` CLI binary and the DuckLake store

In [ ]:
import json
import os
import pathlib
import subprocess

import ailake
import duckdb
import numpy as np

AILAKE_BIN = "ailake"
STORE      = os.environ.get("DEMO_DUCKLAKE_STORE", "/data/ailake_ducklake")
DIM        = int(os.environ.get("DEMO_DIM", "32"))
TABLE      = "default.docs"

pathlib.Path(STORE).mkdir(parents=True, exist_ok=True)

version = subprocess.run([AILAKE_BIN, "--version"], capture_output=True, text=True)
print(version.stdout.strip() or version.stderr.strip())
print(f"store = {STORE}")
print(f"dim   = {DIM}")


def run_cli(*args, check=True):
    """Run `ailake --store STORE --catalog ducklake <args>`, print stdout, return CompletedProcess."""
    cmd = [AILAKE_BIN, "--store", STORE, "--catalog", "ducklake", *args]
    print("$", " ".join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
        if check:
            raise RuntimeError(f"ailake {args[0]} failed (exit {proc.returncode})")
    return proc


## 1. `create` — new table on the DuckLake catalog

`--catalog ducklake` only needs a local `--store` — no connection string. First call
does `INSTALL ducklake; LOAD ducklake;` under the hood (network dependency on first
use, cached after), then bootstraps `catalog/ailake_root.db` (sidecar) and
`catalog/ducklake_meta.db` (DuckLake's own metadata) under `STORE`.

In [ ]:
run_cli("create", TABLE, "--dim", str(DIM), "--metric", "cosine",
        "--precision", "f16", "--column", "embedding")

print(sorted(p.name for p in pathlib.Path(STORE, "catalog").glob("*")))


## 2. Seed a valid AI-Lake Parquet file via the Python SDK

`ailake insert` reads its source Parquet's embedding column as `FixedSizeBinary`
(F16-encoded floats) — the same wire format every AI-Lake writer produces, not a
plain `list<float32>`. Easiest way to get a byte-correct source file: write one
small batch through `ailake.TableWriter` against a throwaway path (any catalog —
this uses the default `HadoopCatalog`, unrelated to the DuckLake table above), then
hand the resulting `data/*.parquet` file to the CLI.

In [ ]:
SEED_PATH = "/tmp/ducklake_seed"

texts_a = [
    "HNSW graph traversal for approximate nearest neighbor search.",
    "DuckLake stores catalog metadata in DuckDB instead of Avro manifests.",
    "Geometric pruning skips files whose centroid is far from the query.",
    "Product quantization compresses vectors to a handful of bytes each.",
    "Iceberg snapshots give AI-Lake tables ACID transactions and time travel.",
]
rng = np.random.default_rng(seed=7)
embs_a = rng.standard_normal((len(texts_a), DIM)).astype(np.float32)
embs_a /= np.linalg.norm(embs_a, axis=1, keepdims=True)

w = ailake.TableWriter(SEED_PATH, dim=DIM, metric="cosine")
w.write_batch(texts_a, embs_a.tolist())
w.commit()

seed_file_a = next(pathlib.Path(SEED_PATH, "data").glob("*.parquet"))
print(f"seed file: {seed_file_a}")


## 3. `insert` — load the seed file into the DuckLake table

Real bug caught wiring this up (documented in `DUCKLAKE_CATALOG.md`): the first
version resolved `DataFileEntry::path` (warehouse-relative, e.g.
`data/part-00000.parquet`) against DuckDB's own process working directory instead
of the warehouse root — `ducklake_add_data_files` failed with *"No files found that
match the pattern"* on the very first real insert. Fixed by resolving to an
absolute path only at the SQL call site.

In [ ]:
run_cli("insert", TABLE, str(seed_file_a),
        "--embeddings", "embedding", "--metric", "cosine", "--precision", "f16")


## 4. `search` — vector similarity against the DuckLake-catalog table

In [ ]:
query = embs_a[0].tolist()  # nearest neighbour should be texts_a[0] itself
query_str = ",".join(f"{x:.6f}" for x in query)

proc = run_cli("search", TABLE, "--query", query_str, "--top-k", "5", "--format", "json")
for h in json.loads(proc.stdout)["results"]:
    print(h)


## 5. `evolve` — add a column without rewriting data files

Adds `tag: string` to `metadata.json` via a real `ALTER TABLE ... ADD COLUMN`
against the `lake` attachment. Old rows (inserted in step 3, before this column
existed) read back as `null` for `tag` — no compaction needed.

In [ ]:
run_cli("evolve", TABLE, "--add", "tag:string")


## 6. Insert again — a file older than the new column

`ducklake_add_data_files` needs **both** `ignore_extra_columns => true` (this new
file has columns DuckLake hasn't been told about) and `allow_missing => true` (the
seed file from step 3 predates `tag` and doesn't have it). The second real bug
found wiring this backend: the extension's default `allow_missing => false`
rejected such files outright until both flags were passed together.

In [ ]:
texts_b = [
    "Compaction merges small files into one, rebuilding a single HNSW graph.",
    "Reciprocal rank fusion blends BM25 and vector search scores.",
]
embs_b = rng.standard_normal((len(texts_b), DIM)).astype(np.float32)
embs_b /= np.linalg.norm(embs_b, axis=1, keepdims=True)

w2 = ailake.TableWriter(SEED_PATH, dim=DIM, metric="cosine")
w2.write_batch(texts_b, embs_b.tolist())
w2.commit()

seed_files = sorted(pathlib.Path(SEED_PATH, "data").glob("*.parquet"))
seed_file_b = seed_files[-1]
print(f"seed file: {seed_file_b}")

run_cli("insert", TABLE, str(seed_file_b),
        "--embeddings", "embedding", "--metric", "cosine", "--precision", "f16")


## 7. `compact` — merge the two small files into one

In [ ]:
run_cli("compact", TABLE, "--min-files", "2", "--format", "json")


## 8. `info` — table statistics after compaction

In [ ]:
proc = run_cli("info", TABLE, "--format", "json")
print(json.dumps(json.loads(proc.stdout), indent=2))


## 9. Open the sidecar directly — `main.ailake_vector_index`

The sidecar is a plain DuckDB file (`catalog/ailake_root.db`, no DuckLake
versioning). This is where centroid, radius, HNSW offset/length and index status
live — fields DuckLake's own fixed schema has no room for.

In [ ]:
root_db = str(pathlib.Path(STORE, "catalog", "ailake_root.db"))
con = duckdb.connect(root_db, read_only=True)

df = con.sql('''
    SELECT path, record_count, vector_dim,
           centroid_b64 IS NOT NULL AS has_centroid,
           round(radius, 4) AS radius,
           index_status, active
    FROM ailake_vector_index
    ORDER BY path
''').df()
con.close()
df


## 10. Open the DuckLake attachment directly — real row data

`lake.default.docs` is a genuine DuckLake-managed table. Any DuckLake-aware reader
(bare `duckdb` CLI here, or a future Spark/Trino DuckLake connector) can query it
with no AI-Lake code involved.

**Lazy column declaration** (see `DUCKLAKE_CATALOG.md`): only the primary vector
column is declared to DuckLake at `create_table` time; other columns arrive later
via `evolve` (step 5 added `tag`). The row-text column (physically named `text`
in the Parquet file — check with `pyarrow.parquet.ParquetFile(...).schema_arrow`)
was **never** declared, so it isn't part of `lake.default.docs`'s schema at all —
`DESCRIBE lake.default.docs` below only shows `embedding` and `tag`, confirming
the data is there on disk but not yet DuckLake-queryable.

In [ ]:
con2 = duckdb.connect()
con2.sql("INSTALL ducklake; LOAD ducklake;")
meta_db = str(pathlib.Path(STORE, "catalog", "ducklake_meta.db"))
data_path = str(pathlib.Path(STORE, "data"))
con2.sql(f"ATTACH 'ducklake:{meta_db}' AS lake (DATA_PATH '{data_path}')")

print(con2.sql("SELECT count(*) AS row_count FROM lake.default.docs").df())
print()
print("Declared schema (embedding + the evolved tag column only):")
print(con2.sql("DESCRIBE lake.default.docs").df())
print()
print(con2.sql("SELECT tag FROM lake.default.docs ORDER BY tag").df())
con2.close()


## 11. Known v1 limitations

- **Local filesystem only** — `--catalog ducklake` rejects `s3://`/`gs://`/`az://` stores.
- **No `ailake-py` binding** — CLI-only; that's why this notebook shells out instead
  of `import ailake` for the catalog operations (steps 1, 3, 5–8).
- **Single-writer** — the metadata catalog is a local DuckDB file (SQLite-class
  constraint); concurrent writers to the same table are not safe.
- **Retired files aren't physically reclaimed** — compaction/overwrite issue a real
  `DELETE FROM lake.tbl WHERE filename = ?` (any DuckLake reader sees the row gone
  immediately), but the file itself stays registered until an operator runs
  DuckLake's own `ducklake_expire_snapshots`/`ducklake_cleanup_files`. Not wired
  into this backend.
- **Lazy column declaration** — demonstrated live in step 10: the row-text
  column is on disk but not in `lake.default.docs`'s schema until an `evolve`
  call declares it.
- **`JdbcCatalog`/`GlueCatalog`/`NessieCatalog`** remain backend-only (implemented,
  tested, no CLI/`ailake-py` selector) — unrelated gaps, not something this
  backend's CLI wiring implies is now easy for the others.

See `docs/guides/DUCKLAKE_CATALOG.md` for the full design writeup, including three
real bugs (two in this backend, one pre-existing in the shared Parquet-precision
read path) found verifying it against a live `ducklake` extension.